# 🎮 Amazon Video Games — Exploratory Data Analysis

This notebook performs a structured EDA on the Amazon Video Games ratings dataset to uncover patterns that directly inform the design choices of our recommendation system.

**Dataset files used:**
- `../data/raw/video_games_reviews.csv` — raw ratings (user_id, item_id, rating, timestamp)
- `../data/processed/ratings_filtered.csv` — filtered ratings
- `../data/processed/train.csv`, `val.csv`, `test.csv` — train/val/test splits

---

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Apply dark background style globally
plt.style.use('dark_background')

# ---------- Palette ----------
PURPLE    = '#764ba2'
BLUE      = '#667eea'
GREEN     = '#55A868'
ACCENT    = '#f093fb'
BG_DARK   = '#1a1a2e'

# ---------- Paths (relative to recsys/) ----------
RAW_PATH       = '../data/raw/video_games_reviews.csv'
FILTERED_PATH  = '../data/processed/ratings_filtered.csv'
TRAIN_PATH     = '../data/processed/train.csv'
VAL_PATH       = '../data/processed/val.csv'
TEST_PATH      = '../data/processed/test.csv'

# ---------- Load data ----------
raw_df      = pd.read_csv(RAW_PATH)
filtered_df = pd.read_csv(FILTERED_PATH)
train_df    = pd.read_csv(TRAIN_PATH)
val_df      = pd.read_csv(VAL_PATH)
test_df     = pd.read_csv(TEST_PATH)

print('Raw reviews       :', raw_df.shape)
print('Filtered ratings  :', filtered_df.shape)
print('Train / Val / Test:', train_df.shape, val_df.shape, test_df.shape)
raw_df.head(3)

---
## Section 1 — Rating Distribution

### What we're looking at
We plot the raw count of reviews for each star rating (1 – 5) in the **unfiltered** dataset.

### Why it matters for recommendation systems
Rating distributions shape every metric we use. If ~64 % of all ratings are 5-stars, a naïve model that predicts **5 for every item** achieves a surprisingly low RMSE — not because it is good, but because the distribution is so lopsided. Understanding this skew helps us:
- Choose **bias-aware** loss functions (e.g., normalised RMSE or ranking metrics like NDCG).
- Implement **user/item bias terms** in matrix factorisation models.
- Avoid optimistic RMSE baselines that mask real recommendation quality.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5), facecolor=BG_DARK)
ax.set_facecolor(BG_DARK)

rating_counts = raw_df['rating'].value_counts().sort_index()
rating_pct    = (rating_counts / rating_counts.sum() * 100).round(1)

bars = ax.bar(
    rating_counts.index,
    rating_counts.values,
    color=[PURPLE, PURPLE, BLUE, BLUE, '#f093fb'],
    edgecolor='white',
    linewidth=0.5,
    width=0.6,
    zorder=3
)

# Annotate bars with percentage
for bar, pct in zip(bars, rating_pct.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + rating_counts.max() * 0.015,
        f'{pct}%',
        ha='center', va='bottom',
        color='white', fontsize=11, fontweight='bold'
    )

ax.set_xlabel('Star Rating', color='white', fontsize=13)
ax.set_ylabel('Number of Reviews', color='white', fontsize=13)
ax.set_title('Rating Distribution — Raw Amazon Video Games Dataset', color='white', fontsize=15, pad=15)
ax.set_xticks([1, 2, 3, 4, 5])
ax.tick_params(colors='white')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(axis='y', linestyle='--', alpha=0.25, zorder=0)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

### Key Observations
- **Extreme positive skew**: 5-star ratings dominate the dataset (~64 % of all reviews), making the average rating artificially high and unrepresentative of nuanced user preferences.
- **RMSE is misleading**: A trivial model that always predicts 5.0 would achieve a low RMSE because the mode is 5 — this is why ranking metrics (NDCG, Precision@K) are preferred alongside RMSE.
- **Negative feedback is rare**: 1- and 2-star ratings together represent less than 10 % of all reviews, suggesting users on Amazon tend to self-select products they already expect to like (purchase-bias).

---
## Section 2 — User Activity Distribution

### What we're looking at
We plot a **log-scale histogram** of the number of ratings each user has contributed to the filtered dataset.

### Why it matters for recommendation systems
User activity follows a **power-law (Pareto) distribution** — a hallmark of real-world collaborative filtering datasets. This means:
- The majority of users have rated only 1–5 items, providing almost no signal for learning their preferences.
- A small fraction of "power users" account for a disproportionate share of all ratings.
- Cold-start is the norm, not the exception — CF models degrade badly for low-activity users, motivating **hybrid approaches** (content + CF) and **popularity-based fallbacks**.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), facecolor=BG_DARK)
ax.set_facecolor(BG_DARK)

user_activity = filtered_df.groupby('user_id').size()

# Log-spaced bins for power-law visibility
bins = np.logspace(
    np.log10(user_activity.min()),
    np.log10(user_activity.max()),
    50
)

ax.hist(
    user_activity.values,
    bins=bins,
    color=BLUE,
    edgecolor='white',
    linewidth=0.3,
    alpha=0.85,
    zorder=3
)

ax.set_xscale('log')
ax.set_yscale('log')

# Median line
median_activity = user_activity.median()
ax.axvline(median_activity, color=ACCENT, linewidth=1.8, linestyle='--', label=f'Median = {median_activity:.0f} ratings')
ax.legend(fontsize=11, framealpha=0.2)

ax.set_xlabel('Number of Ratings per User (log scale)', color='white', fontsize=13)
ax.set_ylabel('Number of Users (log scale)', color='white', fontsize=13)
ax.set_title('User Activity Distribution — Power-Law Behaviour', color='white', fontsize=15, pad=15)
ax.tick_params(colors='white')
ax.grid(which='both', linestyle='--', alpha=0.2, zorder=0)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

### Key Observations
- **Power-law shape**: The histogram is linear on a log-log scale, confirming the classical power-law (Pareto) distribution — a small minority of users generate most of the ratings.
- **Sparse signal for most users**: The median user has rated very few items, making it nearly impossible for a pure collaborative filter to learn reliable preference vectors without regularisation or augmentation.
- **Filtering threshold matters**: Removing users below a minimum activity threshold (e.g., 5 ratings) is essential for model stability, but must be done carefully to avoid discarding too much of the user base.

---
## Section 3 — Item Popularity Distribution

### What we're looking at
We plot a **log-scale histogram** of the number of ratings each item has received in the filtered dataset.

### Why it matters for recommendation systems
The item-side analogue of user activity reveals the **long-tail problem** — one of the most important challenges in RecSys:
- Most items have only a handful of ratings, making it hard to learn reliable item embeddings (**item cold-start**).
- Popular items get recommended repeatedly, creating a feedback loop that buries niche items even when they might be a perfect fit.
- Addressing the long tail is critical for **novelty**, **diversity**, and **fairness** in recommendations.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), facecolor=BG_DARK)
ax.set_facecolor(BG_DARK)

item_popularity = filtered_df.groupby('item_id').size()

bins = np.logspace(
    np.log10(max(item_popularity.min(), 1)),
    np.log10(item_popularity.max()),
    50
)

ax.hist(
    item_popularity.values,
    bins=bins,
    color=GREEN,
    edgecolor='white',
    linewidth=0.3,
    alpha=0.85,
    zorder=3
)

ax.set_xscale('log')
ax.set_yscale('log')

# Median line
median_pop = item_popularity.median()
ax.axvline(median_pop, color=ACCENT, linewidth=1.8, linestyle='--', label=f'Median = {median_pop:.0f} ratings')
ax.legend(fontsize=11, framealpha=0.2)

ax.set_xlabel('Number of Ratings per Item (log scale)', color='white', fontsize=13)
ax.set_ylabel('Number of Items (log scale)', color='white', fontsize=13)
ax.set_title('Item Popularity Distribution — The Long-Tail Problem', color='white', fontsize=15, pad=15)
ax.tick_params(colors='white')
ax.grid(which='both', linestyle='--', alpha=0.2, zorder=0)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

### Key Observations
- **Long-tail distribution**: The vast majority of items have very few ratings — this is the canonical long-tail effect that makes purely collaborative methods unreliable for most of the catalogue.
- **Item cold-start is pervasive**: Many items have so few ratings that latent factor models cannot learn meaningful embeddings; content-based signals (item metadata, description) are needed as a fallback.
- **Popularity bias risk**: Recommending only well-rated popular items ignores the long tail where niche but highly relevant items live — diversity metrics and tail-aware losses help mitigate this.

---
## Section 4 — Rating Sparsity Visualisation

### What we're looking at
We sample a **50 × 50 submatrix** of the user-item interaction matrix and render it as a heatmap. A cell is coloured if a rating exists (1), white/grey if not (0).

### Why it matters for recommendation systems
Matrix sparsity is perhaps the defining challenge of collaborative filtering:
- Real-world interaction matrices are typically **99 %+ sparse** — most users have rated a vanishingly small fraction of all items.
- This sparsity is why we need **latent factor models** (SVD, ALS, neural CF) that generalise from observed entries to unobserved ones.
- Visualising sparsity makes it immediately intuitive *why* simple memory-based methods (nearest-neighbour CF) scale poorly — there are almost no shared ratings between any two users.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8), facecolor=BG_DARK)
ax.set_facecolor(BG_DARK)

SAMPLE_SIZE = 50

# Pick the top-50 most active users and top-50 most popular items
top_users = (
    filtered_df.groupby('user_id').size()
    .nlargest(SAMPLE_SIZE).index.tolist()
)
top_items = (
    filtered_df.groupby('item_id').size()
    .nlargest(SAMPLE_SIZE).index.tolist()
)

sample_df = filtered_df[
    filtered_df['user_id'].isin(top_users) &
    filtered_df['item_id'].isin(top_items)
].copy()

# Build binary interaction matrix
matrix = (
    sample_df
    .drop_duplicates(subset=['user_id', 'item_id'])
    .pivot(index='user_id', columns='item_id', values='rating')
    .reindex(index=top_users, columns=top_items)
    .notna()
    .astype(int)
)

sparsity = 1 - matrix.values.mean()

# Custom colourmap: 0 = dark, 1 = purple/blue
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('sparse', ['#0d0d1a', BLUE])

sns.heatmap(
    matrix,
    ax=ax,
    cmap=cmap,
    cbar=False,
    linewidths=0.3,
    linecolor='#1a1a2e',
    xticklabels=False,
    yticklabels=False
)

ax.set_title(
    f'User–Item Interaction Matrix (50 × 50 sample)\nSparsity = {sparsity:.1%}',
    color='white', fontsize=14, pad=15
)
ax.set_xlabel('Top-50 Items (by popularity)', color='white', fontsize=12)
ax.set_ylabel('Top-50 Users (by activity)', color='white', fontsize=12)
ax.tick_params(colors='white')

plt.tight_layout()
plt.show()

# Global sparsity across the full filtered set
n_users = filtered_df['user_id'].nunique()
n_items = filtered_df['item_id'].nunique()
n_ratings = len(filtered_df)
global_sparsity = 1 - n_ratings / (n_users * n_items)
print(f'Global matrix sparsity: {global_sparsity:.4%}')
print(f'Users: {n_users:,}  |  Items: {n_items:,}  |  Ratings: {n_ratings:,}')

### Key Observations
- **Visually striking sparsity**: Even among the *most active* users and *most popular* items, the heatmap is dominated by empty cells — the global sparsity of the full matrix typically exceeds 99.9 %.
- **Latent factors are essential**: No neighbourhood-based method can reliably find similar users when so few items are co-rated; matrix factorisation and neural methods that generalise via continuous embeddings are far better suited.
- **Evaluation must account for sparsity**: Precision@K and Recall@K metrics should be computed only on items actually present in the test set, since the vast majority of "unseen" items were simply never rated rather than being disliked.

---
## Section 5 — Temporal Rating Volume

### What we're looking at
We aggregate all ratings by **calendar year** (extracted from the `timestamp` column) and plot a line chart showing how review volume grew over time.

### Why it matters for recommendation systems
The temporal structure of ratings has several implications:
- **Temporal train/val/test splits** are more realistic than random splits — otherwise we inadvertently use future data to predict the past (data leakage).
- Rapid growth in recent years means the **distribution of the test set** can differ significantly from the training set (concept drift).
- Models that incorporate **recency weighting** or **time-aware embeddings** can leverage this structure for better predictions.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5), facecolor=BG_DARK)
ax.set_facecolor(BG_DARK)

ts = raw_df['timestamp'].copy()

# Detect whether timestamps are in milliseconds or seconds
# Unix ms timestamps are typically > 1e11; seconds are ~1e9
if ts.median() > 1e11:
    ts = ts / 1000  # convert ms → seconds

years = pd.to_datetime(ts, unit='s', errors='coerce').dt.year
years = years.dropna().astype(int)

# Keep only plausible years (2000-2024)
years = years[(years >= 2000) & (years <= 2024)]

yearly_counts = years.value_counts().sort_index()

# Gradient fill under the line
ax.fill_between(
    yearly_counts.index,
    yearly_counts.values,
    alpha=0.25,
    color=BLUE
)
ax.plot(
    yearly_counts.index,
    yearly_counts.values,
    color=BLUE,
    linewidth=2.5,
    marker='o',
    markersize=6,
    markerfacecolor=ACCENT,
    markeredgecolor='white',
    markeredgewidth=0.8,
    zorder=4
)

# Annotate peak year
peak_year = yearly_counts.idxmax()
peak_val  = yearly_counts.max()
ax.annotate(
    f'Peak: {peak_year}\n{peak_val:,} reviews',
    xy=(peak_year, peak_val),
    xytext=(peak_year - 2, peak_val * 0.80),
    arrowprops=dict(arrowstyle='->', color=ACCENT, lw=1.5),
    color=ACCENT,
    fontsize=10,
    ha='center'
)

ax.set_xlabel('Year', color='white', fontsize=13)
ax.set_ylabel('Number of Reviews', color='white', fontsize=13)
ax.set_title('Temporal Review Volume — Amazon Video Games', color='white', fontsize=15, pad=15)
ax.tick_params(colors='white')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_xticks(yearly_counts.index)
ax.set_xticklabels(yearly_counts.index, rotation=45, ha='right')
ax.grid(axis='y', linestyle='--', alpha=0.2, zorder=0)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

### Key Observations
- **Exponential growth**: Review volume grew dramatically over the years, reflecting both Amazon's expanding user base and the rising popularity of gaming — early years have very few ratings, which can destabilise time-based train/val splits if the cutoff is set too early.
- **Temporal leakage risk**: Random train/val/test splits ignore this chronological structure; a model trained on random 80 % of data may inadvertently use reviews from 2018 to predict a review written in 2015, inflating offline metrics but degrading real-world performance.
- **Concept drift**: User preferences and popular game genres shift over time; models trained only on older data may not generalise to the recommendation context of peak-activity years, motivating recency-weighted training or sliding-window retraining strategies.

---
## Summary of EDA Findings

| # | Analysis | Key Finding | Modelling Implication |
|---|----------|-------------|----------------------|
| 1 | Rating Distribution | ~64 % are 5-stars | Use ranking metrics (NDCG, P@K) alongside RMSE; add bias terms |
| 2 | User Activity | Power-law — most users rate ≤ 5 items | Regularisation, minimum activity filters, hybrid fallback |
| 3 | Item Popularity | Long tail — most items have few ratings | Content features for cold-start; tail-aware sampling |
| 4 | Sparsity | Matrix is >99.9 % empty | Latent-factor models (MF, NCF) essential over memory-based CF |
| 5 | Temporal Volume | Exponential growth over time | Chronological splits, recency weighting, concept-drift awareness |

> These findings directly motivate the **Neural Collaborative Filtering + content features** architecture used in this project.